# STEP 8: LLM Router와 LangGraph Conditional Edge

이 노트북에서는 사용자의 자연어 질문을 로컬 LLM이 `basic` 또는 `margin`으로 구조화하고, LangGraph가 그 결과에 따라 다른 경로를 선택하게 합니다.

`사용자 질문 → LLM Router Node → analysis_type State → Conditional Edge → 선택된 경로`

이번 단계에서는 Router의 역할만 보기 위해 OpenDART API, LangChain Tool, Agent는 사용하지 않습니다.

## 0. 이번 단계에서 바뀌는 점

STEP 6에서는 사람이 `analysis_type`을 직접 넣었습니다. 이제는 LLM Router Node가 질문을 읽고 이 값을 반환합니다.

- `basic`: 매출액, 영업이익, 당기순이익 같은 기본 재무 수치 조회
- `margin`: 영업이익률 계산 또는 분석

구조화된 출력 형식이 맞는다고 해서 LLM의 분류가 항상 옳다는 뜻은 아닙니다. Router 정확도 평가는 여러 질문과 정답을 준비하는 별도 단계에서 진행합니다.

## 1. 필요한 클래스 불러오기

- `RouteDecision`: LLM이 반환해야 할 Router 결과의 형태
- `ChatOllama`: 로컬 Ollama 모델 연결
- `StateGraph`: Router 결과를 따라 경로를 선택할 Graph

In [1]:
from typing import Literal, TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

  from .autonotebook import tqdm as notebook_tqdm


## 2. LLM Router의 출력 형식 정의

`Literal`로 가능한 경로를 두 개로 제한합니다. 이 schema는 LLM에게 어떤 형태로 답해야 하는지 알려 주고, Python이 받은 값을 검증하는 기준이 됩니다.

In [2]:
class RouteDecision(BaseModel):
    analysis_type: Literal["basic", "margin"] = Field(
        description="basic은 기본 재무 수치 조회, margin은 영업이익률 계산 또는 분석"
    )

## 3. 로컬 LLM과 구조화된 Router 준비

`with_structured_output(RouteDecision)`은 LLM의 일반 문장 답변 대신 `RouteDecision` 형태의 결과를 요청합니다. `temperature=0`은 같은 입력에서 결과 변동을 줄이기 위한 설정입니다.

In [3]:
llm = ChatOllama(
    model="qwen3.6:27b",
    base_url="http://localhost:11434",
    temperature=0,
)

structured_router = llm.with_structured_output(RouteDecision)

## 4. Graph State와 LLM Router Node 정의

`route_question`은 LangGraph Node이면서 내부에서 LLM을 호출합니다. LLM이 반환한 `analysis_type`만 State에 추가합니다.

아래 system message의 분류 규칙은 Prompt입니다. 이후 평가 단계에서는 이 Prompt를 바꿔 Router 정확도를 비교할 수 있습니다.

In [4]:
class RouterState(TypedDict, total=False):
    question: str
    analysis_type: Literal["basic", "margin"]
    selected_path: str


def route_question(state: RouterState) -> dict[str, Literal["basic", "margin"]]:
    messages = [
        SystemMessage(
            content=(
                "기업 재무 질문을 분류한다. 매출액, 영업이익, 당기순이익 같은 "
                "기본 재무 수치 조회는 basic이다. 영업이익률의 계산 또는 분석은 margin이다. "
                "두 종류가 함께 있으면 margin을 선택한다."
            )
        ),
        HumanMessage(content=state["question"]),
    ]
    decision = structured_router.invoke(messages)
    print("LLM Router 결과:", decision.model_dump())
    return {"analysis_type": decision.analysis_type}

## 5. Conditional Edge와 경로 Node 정의

조건 함수는 LLM을 호출하지 않습니다. Router Node가 이미 State에 넣은 `analysis_type`을 읽고 다음 Node 이름만 반환합니다.

두 경로 Node는 이번에는 실제 계산·조회 대신 어떤 경로가 선택됐는지만 State에 기록합니다.

In [5]:
def select_path(
    state: RouterState,
) -> Literal["basic_path", "margin_path"]:
    if state["analysis_type"] == "basic":
        return "basic_path"
    return "margin_path"


def mark_basic_path(state: RouterState) -> dict[str, str]:
    return {"selected_path": "basic 재무 조회 경로"}


def mark_margin_path(state: RouterState) -> dict[str, str]:
    return {"selected_path": "영업이익률 분석 경로"}

## 6. LLM Router를 포함한 Graph 만들기

`route_question` Node가 먼저 LLM을 호출합니다. 그 결과가 `analysis_type`으로 State에 추가된 뒤 Conditional Edge가 다음 경로를 결정합니다.

In [6]:
workflow = StateGraph(RouterState)

workflow.add_node("route_question", route_question)
workflow.add_node("basic_path", mark_basic_path)
workflow.add_node("margin_path", mark_margin_path)

workflow.add_edge(START, "route_question")
workflow.add_conditional_edges(
    "route_question",
    select_path,
    {
        "basic_path": "basic_path",
        "margin_path": "margin_path",
    },
)
workflow.add_edge("basic_path", END)
workflow.add_edge("margin_path", END)

app = workflow.compile()

## 7. Graph 구조 확인

Mermaid 텍스트에서 `route_question` 뒤에 두 경로가 있는지 확인합니다. 어떤 경로를 선택할지는 실행 시 LLM Router 결과에 따라 달라집니다.

In [7]:
print(app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	route_question(route_question)
	basic_path(basic_path)
	margin_path(margin_path)
	__end__([<p>__end__</p>]):::last
	__start__ --> route_question;
	route_question -.-> basic_path;
	route_question -.-> margin_path;
	basic_path --> __end__;
	margin_path --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 8. 기본 재무 조회 질문 실행

질문을 바꿔 보고, 출력된 LLM Router 결과와 최종 `selected_path`를 함께 확인합니다.

In [8]:
basic_question = "삼성전자의 매출액과 영업이익을 알려줘."
basic_result = app.invoke({"question": basic_question})
basic_result

LLM Router 결과: {'analysis_type': 'basic'}


{'question': '삼성전자의 매출액과 영업이익을 알려줘.',
 'analysis_type': 'basic',
 'selected_path': 'basic 재무 조회 경로'}

## 9. 영업이익률 질문 실행

이번에는 같은 Graph에 다른 질문만 전달합니다. LLM Router가 `margin`을 반환하면 Conditional Edge가 `margin_path`로 이동합니다.

In [9]:
margin_question = "삼성전자의 영업이익률을 분석해줘."
margin_result = app.invoke({"question": margin_question})
margin_result

LLM Router 결과: {'analysis_type': 'margin'}


{'question': '삼성전자의 영업이익률을 분석해줘.',
 'analysis_type': 'margin',
 'selected_path': '영업이익률 분석 경로'}

## 10. 최소 확인

이 확인은 Router의 정량 평가가 아니라, 두 예시 질문이 구조화된 값과 Graph 분기로 연결됐는지 보는 학습용 점검입니다.

In [10]:
assert basic_result["analysis_type"] == "basic"
assert basic_result["selected_path"] == "basic 재무 조회 경로"
assert margin_result["analysis_type"] == "margin"
assert margin_result["selected_path"] == "영업이익률 분석 경로"

print("LLM Router 결과가 Conditional Edge의 분기값으로 사용되었습니다.")

LLM Router 결과가 Conditional Edge의 분기값으로 사용되었습니다.


## 이번 단계에서 기억할 점

- LLM Router는 자연어 질문을 `analysis_type` 같은 구조화된 값으로 바꿉니다.
- LangGraph는 그 값을 보고 Edge를 통해 다음 Node를 선택합니다.
- 구조화된 출력은 값의 **형식**을 제한하지만, 분류의 **의미적 정확성**까지 보장하지는 않습니다.
- 나중의 Router 평가는 고정 질문셋과 정답 route를 기준으로 Prompt·모델·schema가 정확도와 속도에 미치는 영향을 비교합니다.